In [1]:
# ======================================
# STEP 1: Load ENV
# ======================================
import os
from dotenv import load_dotenv

load_dotenv()

NVIDIA_API_KEY = os.getenv("NVIDIA_API_KEY")


# ======================================
# STEP 2: NVIDIA LLM
# ======================================
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

llm = ChatOpenAI(

    api_key=NVIDIA_API_KEY,

    base_url="https://integrate.api.nvidia.com/v1",

    model="meta/llama3-70b-instruct",

    temperature=0
)


# ======================================
# STEP 3: Dummy University DB
# ======================================
students = {

    "S101": {

        "name": "Rahul",

        "attendance": "85%",

        "results": "Math A | Physics B+"
    },

    "S102": {

        "name": "Priya",

        "attendance": "92%",

        "results": "Math A+ | Physics A"
    }

}


# ======================================
# STEP 4: MCP Tools
# ======================================
def view_attendance(student_id):

    if student_id in students:

        return f"📊 Attendance: {students[student_id]['attendance']}"

    return "❌ Student not found"



def view_results(student_id):

    if student_id in students:

        return f"📝 Results: {students[student_id]['results']}"

    return "❌ Student not found"



def update_results(student_id):

    if student_id in students:

        return "✅ Exam results updated"

    return "❌ Student not found"



def approve_course():

    return "🎓 Course change approved"



def view_all_records():

    return f"📚 Records: {students}"


# ======================================
# STEP 5: RBAC Security
# ======================================
def secure_access(

    role,

    user_id,

    requested_id,

    action

):

    if role == "admin":

        return True


    if role == "faculty":

        return action in [

            "view_attendance",

            "view_results",

            "update_results",

            "view_all_records"

        ]


    if role == "student":

        return (

            user_id == requested_id

            and action in [

                "view_attendance",

                "view_results"

            ]

        )

    return False


# ======================================
# STEP 6: Intent detection via LLM
# ======================================
def decide_action(query):

    prompt = f"""

    Classify into ONE action:

    view_attendance
    view_results
    update_results
    approve_course
    view_all_records

    Query: {query}

    Return only action name.
    """

    response = llm.invoke(

        [HumanMessage(content=prompt)]

    )

    return response.content.strip().lower()


# ======================================
# STEP 7: MCP Agent
# ======================================
def university_agent(

    message,

    role,

    user_id,

    requested_id,

    history

):

    if history is None:

        history = []


    if not user_id:

        reply = "⚠️ enter id"

        history.append({

            "role": "assistant",

            "content": reply
        })

        return "", history


    if not requested_id:

        requested_id = user_id


    action = decide_action(message)


    if not secure_access(

        role,

        user_id,

        requested_id,

        action
    ):

        reply = "🚫 Access denied"


    else:

        if action == "view_attendance":

            reply = view_attendance(requested_id)


        elif action == "view_results":

            reply = view_results(requested_id)


        elif action == "update_results":

            reply = update_results(requested_id)


        elif action == "approve_course":

            reply = approve_course()


        elif action == "view_all_records":

            reply = view_all_records()


        else:

            reply = "🤖 ask academic question"


    history.append({

        "role": "user",

        "content": message
    })


    history.append({

        "role": "assistant",

        "content": reply
    })


    return "", history


# ======================================
# STEP 8: Gradio UI
# ======================================
import gradio as gr


with gr.Blocks() as demo:

    gr.Markdown("## 🏫 University Smart Assistant (RBAC + MCP)")


    role = gr.Dropdown(

        ["student", "faculty", "admin"],

        value="student",

        label="Role"
    )


    user_id = gr.Textbox(

        value="S101",

        label="Your ID"
    )


    requested_id = gr.Textbox(

        label="Target Student ID (optional)"
    )


    chatbot = gr.Chatbot(height=400)


    msg = gr.Textbox(

        label="Ask question"
    )


    state = gr.State([])


    msg.submit(

        university_agent,

        inputs=[

            msg,

            role,

            user_id,

            requested_id,

            state

        ],

        outputs=[msg, chatbot]

    )


demo.launch()

* Running on local URL:  http://127.0.0.1:7868
* To create a public link, set `share=True` in `launch()`.
